# **LASSO regression**

Marek Šugár

In this notebook we aim to optimize regularization parameter $\lambda$ for every single stock ticker included in database. Using this we can optimize the overall performance more however, needs to be strongly checked on testing as we are balancing on the edge of overfit, of course.

In [4]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from DataFramePrep import generate_TrainingDataFrame

# Metrics to measure successfulness
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler

# ML Stuff
from sklearn.linear_model import Lasso

# In case of convergence problem, supress warning
import warnings

warnings.filterwarnings("ignore")

In [5]:
TrainingDataFrame, tickers, historic_columns, StockDataDatabase = generate_TrainingDataFrame()

# **LASSO regression**

In [6]:
performance_tracker_regularization = {}

for ticker in tickers["Ticker"]:
    performance_tracker_regularization[ticker] = []
    for Alpha in np.arange(4.5, 4.9, 0.01):
        Stock_Data = pd.read_sql(
            f"SELECT * FROM StockData WHERE Ticker='{ticker}' AND Date>='2017-09-14'",
            con=StockDataDatabase, parse_dates=["Date"]
        )
            
        Stock_Data = pd.merge(Stock_Data, TrainingDataFrame, on="Date")
        Stock_Data["Target"] = Stock_Data["Close"]
        Stock_Data = Stock_Data.dropna().reset_index(drop=True)

        training_length = 4
        prediction_length = 1

        MAPEs = []

        for window_start in range(len(Stock_Data) - (training_length + prediction_length)):
            # Define feature and target windows
            Training_Features = Stock_Data[historic_columns].iloc[window_start:window_start+training_length]
            Training_Target = Stock_Data["Target"].iloc[window_start:window_start+training_length]

            Test_Features = Stock_Data[historic_columns].iloc[
                window_start+training_length : window_start+training_length+prediction_length
            ]
            Test_Target = Stock_Data["Target"].iloc[
                window_start+training_length : window_start+training_length+prediction_length
            ]

            # Skip empty slices
            if Training_Features.empty or Test_Features.empty:
                continue
            
            # Scale only selected features
            scaler = StandardScaler()
            Training_Features = scaler.fit_transform(Training_Features)
            Test_Features = scaler.transform(Test_Features)

            # Fit KNN
            MODEL = Lasso(alpha=Alpha)
            MODEL.fit(Training_Features, Training_Target)

            # Predict and evaluate
            prediction = MODEL.predict(Test_Features)
            MAPEs.append(100 * mean_absolute_percentage_error(Test_Target, prediction))
            
        # Save results
        performance_tracker_regularization[ticker].append((Alpha, np.mean(MAPEs)))
        print(Alpha, np.mean(MAPEs))


4.5 1.5776445527429337
4.51 1.577637105660958
4.52 1.577638115111669
4.529999999999999 1.5776218596556923
4.539999999999999 1.5775969949298416
4.549999999999999 1.5775732123649564
4.559999999999999 1.5775481575659578
4.5699999999999985 1.5775258563418044
4.579999999999998 1.5774963526188457
4.589999999999998 1.5774655038446
4.599999999999998 1.577435712674605
4.609999999999998 1.5774065923289944
4.619999999999997 1.5773806118914062
4.629999999999997 1.577353393953737
4.639999999999997 1.5773269562871104
4.649999999999997 1.5773035795788473
4.659999999999997 1.5772804758878347
4.669999999999996 1.5772589278538227
4.679999999999996 1.577230918288839
4.689999999999996 1.5772026105731172
4.699999999999996 1.577176843276067
4.7099999999999955 1.577147722901531
4.719999999999995 1.5771184894393315
4.729999999999995 1.57708974919681
4.739999999999995 1.5770627099877705
4.749999999999995 1.5770350901652594
4.7599999999999945 1.5770688363936134
4.769999999999994 1.5771025810810655
4.77999999999

In [7]:
parameters = performance_tracker_regularization.copy()
optimal = {}

for ticker in tickers["Ticker"]:
    best_ones = sorted(parameters[ticker], key=lambda x: x[1])
    print(ticker, best_ones[0][0])

ACN 4.8999999999999915
ADBE 4.8999999999999915
AMD 4.5
AKAM 4.5
APH 4.8999999999999915
ADI 4.5
AAPL 4.5
AMAT 4.5
ANET 4.8999999999999915
ADSK 4.5
AVGO 4.8999999999999915
CDNS 4.5
CDW 4.5
CSCO 4.5
CTSH 4.5
GLW 4.5
DELL 4.5
ENPH 4.5
EPAM 4.819999999999993
FFIV 4.5
FICO 4.8999999999999915
FSLR 4.5
FTNT 4.8999999999999915
IT 4.609999999999998
GEN 4.5
GDDY 4.5
HPE 4.5
HPQ 4.5
IBM 4.5
INTC 4.5
INTU 4.8999999999999915
JBL 4.5
KEYS 4.599999999999998
KLAC 4.8999999999999915
LRCX 4.5
MCHP 4.5
MU 4.739999999999995
MSFT 4.5
MPWR 4.8999999999999915
MSI 4.589999999999998
NTAP 4.5
NVDA 4.8999999999999915
NXPI 4.5
ON 4.8999999999999915
ORCL 4.5
PANW 4.5
PTC 4.5
QCOM 4.8999999999999915
ROP 4.529999999999999
CRM 4.5
STX 4.5
NOW 4.8999999999999915
SWKS 4.5
SMCI 4.5
SNPS 4.51
TEL 4.5
TDY 4.5
TER 4.5
TXN 4.5
TRMB 4.7099999999999955
TYL 4.8999999999999915
VRSN 4.5
WDC 4.5
WDAY 4.5
ZBRA 4.8999999999999915
